In [1]:
from dotenv import load_dotenv
import os 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv('../../env', override=True)
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT=os.getenv('END_POINT')
MODEL_NAME=os.getenv('MODEL_NAME')
print(AZURE_OPENAI_API_KEY[:10])
print(MODEL_NAME)

AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT=os.getenv('EMB_END_POINT')
EMB_MODEL_NAME=os.getenv('EMB_MODEL_NAME')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['MODEL_API_VERSION'] = os.getenv('MODEL_API_VERSION')
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv('LANGCHAIN_ENDPOINT')
os.environ['LANGCHAIN_TRACING_V2'] = 'false' #true, false
os.environ['LANGCHAIN_PROJECT'] = 'AGENT'

if os.getenv('LANGCHAIN_TRACING_V2') == "true":
    if len(os.getenv('LANGCHAIN_API_KEY')) > 0:
        print('랭스미스로 추적 중입니다 :', os.getenv('LANGSMITH_API_KEY')[:10])
    else:
        print('랭스미스 키가 확인되지 않았습니다.')

3BHTZdVIXx
gpt-5-nano
랭스미스로 추적 중입니다 : lsv2_pt_ef


In [2]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    model=os.environ['MODEL_NAME'],
    azure_deployment = os.environ["MODEL_NAME"],
    azure_endpoint = os.environ["END_POINT"],
    openai_api_version = os.environ["MODEL_API_VERSION"],
    openai_api_key = os.environ["AZURE_OPENAI_API_KEY"],
)

In [3]:
import asyncio
import time
from typing import Any, List

from utils import (
    CUSTOMER_BASE,
    SUPPORT_BASE,
    build_client,
    send_to_support,
    ask_customer_followup,
    save_transcript,
)


In [4]:
print(f"Customer: {CUSTOMER_BASE}, Support: {SUPPORT_BASE}")

Customer: http://127.0.0.1:8010, Support: http://127.0.0.1:8011


In [5]:
!fuser -k 8010/tcp
!fuser -k 8011/tcp

import sys, subprocess
print("using python:", sys.executable)
subprocess.Popen([sys.executable, "a2a_support_server.py"])
subprocess.Popen([sys.executable, "a2a_customer_server.py"])

using python: /anaconda/envs/kt_env/bin/python


<Popen: returncode: None args: ['/anaconda/envs/kt_env/bin/python', 'a2a_cus...>

In [7]:
customer, customer_http = await build_client(CUSTOMER_BASE)
support,  support_http  = await build_client(SUPPORT_BASE)

고객 에이전트가 빌드 되었습니다.
INFO:     127.0.0.1:40626 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
상담원 에이전트가 빌드 되었습니다.
INFO:     127.0.0.1:48954 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK


In [8]:
start_question = "요금제 변경할 때 위약금이 발생하는 기준을 알려주세요."
sup = await send_to_support(support, start_question)
print("상담원 ▶", sup)

INFO:     127.0.0.1:48954 - "POST / HTTP/1.1" 200 OK
상담원 ▶ 다음 기준으로 위약금이 발생합니다. 위약금은 약정기간별 할인반환금 할인율에 따라 산정되며, 계약기간(약정기간)과 약정 이용기간에 따라 달라집니다.

표: 약정기간별 할인반환금 할인율
- 약정기간: 1년
  - 계약 후 ~ 6개월 이내: 0%
  - 계약 후 7~9개월 이내: 20%
  - 계약 후 10~12개월 이내: 130%

- 약정기간: 2년
  - 계약 후 ~ 6개월 이내: 0%
  - 계약 후 7~12개월 이내: 60%
  - 계약 후 13~16개월 이내: 95%
  - 계약 후 17~20개월 이내: 140%
  - 계약 후 21~24개월 이내: 180%

- 약정기간: 3년
  - 계약 후 ~ 6개월 이내: 0%
  - 계약 후 7~12개월 이내: 30%
  - 계약 후 13~16개월 이내: 65%
  - 계약 후 17~20개월 이내: 75%
  - 계약 후 21~24개월 이내: 100%
  - 계약 후 25~28개월 이내: 110%
  - 계약 후 29~32개월 이내: 125%

참고로 이 내용은 “약정기간별 할인반환금 할인율” 표에 기재된 수치입니다.

참조 문서 및 페이지:
- InternetServiceToU.pdf, 페이지 165(표: 할인반환금 할인율)


In [9]:
cust = await ask_customer_followup(customer, sup)
print("고객 ▶", cust)

INFO:     127.0.0.1:43834 - "POST / HTTP/1.1" 200 OK
고객 ▶ 제 약정 기간이 2년이고 현재 계약 시작 후 8개월이 경과했다면 위약금은 어느 구간에 해당하고, 구체적인 금액을 단계별로 계산해 주실 수 있을까요?


In [10]:
sup = await send_to_support(support, start_question)
print("상담원 ▶", sup)

INFO:     127.0.0.1:54042 - "POST / HTTP/1.1" 200 OK
상담원 ▶ 요금제 변경 시의 위약금은 약정기간별 할인반환금의 할인율을 적용해 산정됩니다. 즉, 약정 기간 동안 받으신 할인액의 합산액에 대해 계약 시점의 남은 기간에 따라 정해진 비율을 곱해 위약금을 부과합니다.

구체적 기준(약정 기간별 할인반환금 할인율)
- 1년 약정
  - 계약 후 0~6개월 이내: 0%
  - 계약 후 7~9개월 이내: 20%
  - 계약 후 10~12개월 이내: 130%
- 2년 약정
  - 계약 후 ~6개월 이내: 0%
  - 계약 후 7~12개월 이내: 60%
  - 계약 후 13~16개월 이내: 95%
  - 계약 후 17~20개월 이내: 140%
  - 계약 후 21~24개월 이내: 180%
- 3년 약정
  - 계약 후 ~6개월 이내: 0%
  - 계약 후 7~12개월 이내: 30%
  - 계약 후 13~16개월 이내: 65%
  - 계약 후 17~20개월 이내: 75%
  - 계약 후 21~24개월 이내: 100%
  - 계약 후 25~28개월 이내: 110%
  - 계약 후 29~32개월 이내: 125%

참고: 위약금은 약정 이용기간 동안 제공받은 약정 할인액의 합산금액에 대해 위 표의 할인율을 적용하여 산출됩니다. 관련 내용은 InternetServiceToU.pdf의 “약정기간별 할인반환금 할인율” 표에 명시되어 있습니다.

참고 문서/출처
- InternetServiceToU.pdf, 페이지 165 (약정기간별 할인반환금 할인율 표)

필요하시면 현재 가입 중인 약정 기간과 남은 기간을 알려주시면, 구체적인 위약금 예상액을 예시로 계산해 드리겠습니다.


In [11]:
topics = [
    "와이파이 요금제에 가입하고 싶어요",
    "요금제 변경 시 위약금이 어떻게 되나요?",
    "해외 로밍 데이터 한도가 있나요?",
    "유심을 교체하고 번호를 바꾸고 싶어요",
]
max_rounds = 3
transcript: List[dict] = []

try:
    for topic in topics:
        print(f"\n[TOPIC] {topic}")
        transcript.append({"who": "customer", "text": topic})

        support_reply = await send_to_support(support, topic)
        transcript.append({"who": "support", "text": support_reply})
        print(f" 상담원 ▶ {support_reply[:120]}...")

        for _ in range(1, max_rounds + 1):
            customer_follow = await ask_customer_followup(customer, support_reply)
            transcript.append({"who": "customer", "text": customer_follow})
            print(f" 고객 ▶ {customer_follow[:120]}...")

            if "감사합니다" in customer_follow:
                break

            support_reply = await send_to_support(support, customer_follow)
            transcript.append({"who": "support", "text": support_reply})
            print(f" 상담원 ▶ {support_reply[:120]}...")

        print("[DONE] topic finished.")
        print("=" * 70)
except Exception as e:
    print(f"오류 발생: {e}")

save_transcript(transcript, prefix="a2a-sim")


[TOPIC] 와이파이 요금제에 가입하고 싶어요
INFO:     127.0.0.1:52900 - "POST / HTTP/1.1" 200 OK
 상담원 ▶ 도와드릴 수 있습니다. 다만 현재 대화에 와이파이 요금제에 대한 구체적 약관 내용이 없어 바로 안내드리기 어렵습니다.

가입을 진행하기 전에 확인해 주시면 좋을 정보와 선택사항은 아래와 같습니다.
- 요금제 유형: ...
INFO:     127.0.0.1:55994 - "POST / HTTP/1.1" 200 OK
 고객 ▶ 거주 중인 지역에서 설치 가능 여부와 함께 설치비 및 약정에 따른 월 요금의 대략적 범위를 먼저 알려주실 수 있을까요?...
INFO:     127.0.0.1:40800 - "POST / HTTP/1.1" 200 OK
 상담원 ▶ 도와드리겠습니다. 다만 현재 대화에 거주지 정보나 요청 서비스 종류가 없어 설치 가능 여부와 설치비/월 요금 범위를 확인할 수 없습니다.

확인을 위해 아래 정보를 알려주실 수 있을까요?
- 거주지 주소(동/구, 필...
INFO:     127.0.0.1:39028 - "POST / HTTP/1.1" 200 OK
 고객 ▶ 거주지 주소(동/구 및 필요 시 우편번호), 원하시는 서비스 유형, 설치를 희망하는 시점, 그리고 약정 기간 선호 여부를 한 번에 알려주실 수 있을까요?...
INFO:     127.0.0.1:42940 - "POST / HTTP/1.1" 200 OK
 상담원 ▶ 한 번에 정리해 드리려면 아래 정보를 한꺼번에 보내주시면 됩니다. 우선 아래 항목을 알려 주세요.

- 거주지 주소: 동/구(필요 시 우편번호)
- 원하시는 서비스 유형
- 설치를 희망하는 시점: 날짜 및 가능 시간...
INFO:     127.0.0.1:58420 - "POST / HTTP/1.1" 200 OK
 고객 ▶ 서비스 유형별로 제공 가능한 속도와 요금 예시를 먼저 알려주실 수 있을까요? 제가 구체적으로 어떤 옵션이 있는지 확인한 뒤에 주소와 설치일을 정하고 싶